# GDG UAM ML Contest in Google Colab

Every submission is evaluated in Google Colab, so all teams are compared under the same conditions. Developing here is optional: any machine with Python 3.10+ works. This notebook gets the materials and the data, runs the baseline, and checks your own submission the way we will.

In [2]:
# the materials (safe to run again)
![ -d /content/ml-contest-materials ] || git clone -q https://github.com/GDG-UAM/ml-contest-materials.git /content/ml-contest-materials
%cd /content/ml-contest-materials

/content/ml-contest-materials


In [3]:
# training data (413 MB) and the demo test set (1.7 MB), from the latest release
!curl -sL --create-dirs -o data/train_100k.npz https://github.com/GDG-UAM/ml-contest-materials/releases/latest/download/train_100k.npz
!curl -sL --create-dirs -o data/test_demo.npz https://github.com/GDG-UAM/ml-contest-materials/releases/latest/download/test_demo.npz

In [4]:
# the baseline: train it, then evaluate it like the real thing, and on the demo test set
!python examples/baseline_train.py
!python harness.py evaluate --model examples/baseline_model.py
!python harness.py evaluate --model examples/baseline_model.py --source file:data/test_demo.npz

features: 101 per sample for 100000 samples (66 s)
validation accuracy: 0.821 on 20000 samples
saved /content/ml-contest-materials/examples/baseline_model.pkl
evaluating baseline_model (examples/baseline_model.py)
Traceback (most recent call last):
  File "/content/ml-contest-materials/harness.py", line 1155, in <module>
    main()
    ~~~~^^
  File "/content/ml-contest-materials/harness.py", line 1149, in main
    evaluate(a.model, resolve_eval_config(overrides), a.out)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ml-contest-materials/harness.py", line 1079, in evaluate
    summaries.append(evaluate_project(path, ecfg, name, out_dir, src, verbose)[0])
                     ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ml-contest-materials/harness.py", line 1007, in evaluate_project
    res = run_once(project, model_path, r, ecfg, src, pdir, status)
  File "/content/ml-contest-materials/harness.py", line 971, in run_once
    

In [6]:
from google.colab import userdata, _message
import os, subprocess, json

os.environ['GI'] = userdata.get('GI')
!git config --global user.email "christianmenhem23@example.com"
!git config --global user.name "Christian-m23"

In [ ]:

TOKEN = userdata.get("GI")
REPO  = "github.com/Christian-m23/ml-contest-gdguam.git"
DIR   = "/content/repo"
URL   = f"https://{TOKEN}@{REPO}"

# clone or update
if not os.path.isdir(DIR):
    subprocess.run(["git", "clone", URL, DIR], check=True)
else:
    subprocess.run(["git", "-C", DIR, "pull", "--rebase"], check=True)

# grab the CURRENT notebook (with all your edits) from Colab's memory
resp = _message.blocking_request("get_ipynb", request="")
nb = resp["ipynb"]
if isinstance(nb, str):
    nb = json.loads(nb)
with open(f"{DIR}/colab.ipynb", "w") as f:
    json.dump(nb, f, indent=1)

# push
subprocess.run(["git", "-C", DIR, "add", "-A"], check=True)
subprocess.run(["git", "-C", DIR, "commit", "-m", "sync from colab"], check=False)
subprocess.run(["git", "-C", DIR, "push", URL, "main"], check=True)
print("✅ pushed: https://github.com/Christian-m23/ml-contest-gdguam")

In [1]:
import numpy as np
from competition_api import load_dataset

train = load_dataset("data/train_100k.npz")
test  = load_dataset("data/test_demo.npz")

def baseline_meta_features(meta):
    """Exact copy of examples/baseline_model.meta_features, for emulation."""
    return [float(v) for _, v in sorted(meta.items()) if isinstance(v, (int, float))]

def baseline_keeps(arr, k=500):
    """Emulate: does meta_features() keep this field? Samples rows to stay cheap."""
    idx = np.linspace(0, len(arr) - 1, min(k, len(arr))).astype(int)
    for i in idx:
        v = arr[i].item() if hasattr(arr[i], "item") else arr[i]
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            return False
    return True

def schema_report(name, ds):
    print("=" * 76)
    print(f"SCHEMA — {name}")
    print("=" * 76)

    s, lab, meta = ds.series, ds.labels, ds.meta

    print("SHAPES / DTYPES")
    print(f"  n samples        : {len(ds)}")
    print(f"  series           : shape={s.shape}  dtype={s.dtype}  writable={s.flags.writeable}  mem={s.nbytes/1e6:.1f} MB")
    print(f"  labels           : shape={lab.shape}  dtype={lab.dtype}  range=[{lab.min()}, {lab.max()}]")
    print(f"  ds[0] type       : {type(ds[0]).__name__}")
    print(f"  ds[0].series     : shape={ds[0].series.shape}  dtype={ds[0].series.dtype}  writable={ds[0].series.flags.writeable}")

    print("\nCLASS COUNTS")
    lab_ok = lab[lab >= 0]
    counts = np.bincount(lab_ok, minlength=8) if len(lab_ok) else np.zeros(8, int)
    for k, c in enumerate(counts):
        pct = c / max(len(lab_ok), 1) * 100
        print(f"  class {k}: {c:>7}  ({pct:5.2f}%)")
    print(f"  sum      : {counts.sum():>7}   unlabeled (-1): {(lab < 0).sum()}")

    print("\nNaN")
    nan_overall = np.isnan(s).mean() * 100
    per_ch = np.isnan(s).mean(axis=(0, 2)) * 100
    per_s  = np.isnan(s).reshape(len(ds), -1).sum(1)
    print(f"  overall rate     : {nan_overall:.3f}%")
    print(f"  per channel      : " + "  ".join(f"ch{c}={r:.3f}%" for c, r in enumerate(per_ch)))
    print(f"  per sample       : mean={per_s.mean():.1f}  median={np.median(per_s):.0f}  max={per_s.max()}")
    print(f"  fully clean      : {(per_s == 0).mean() * 100:.1f}% of samples")

    print("\nMETA")
    for key in sorted(meta):
        arr = meta[key]
        n_unique = len(np.unique(arr))
        first10 = np.unique(arr)[:10].tolist()
        kept = baseline_keeps(arr)
        print(f"  {key!r}")
        print(f"    array dtype    : {arr.dtype}")
        print(f"    unique values  : {n_unique}   first 10: {first10}")
        print(f"    baseline keeps : {kept}")

schema_report("TRAIN", train)
schema_report("DEMO TEST", test)

print("\n" + "=" * 76)
print("BASELINE meta_features() — WHAT ACTUALLY ENTERS THE MODEL")
print("=" * 76)
for name, ds in [("TRAIN", train), ("DEMO TEST", test)]:
    sample = ds[0]
    kept = baseline_meta_features(sample.meta)
    dropped = [k for k, v in sorted(sample.meta.items()) if isinstance(v, bool) or not isinstance(v, (int, float))]
    print(f"\n{name}")
    print(f"  x.meta keys              : {sorted(sample.meta)}")
    print(f"  x.meta values (sample 0) : {sample.meta}")
    print(f"  meta_features() output   : {kept}   (length {len(kept)})")
    print(f"  silently dropped         : {dropped}")
    print(f"  total feature vector     : 92 (per-channel) + 6 (cross-channel) + {len(kept)} (meta) + 1 (NaN frac) = {92 + 6 + len(kept) + 1}")

ModuleNotFoundError: No module named 'competition_api'

In [ ]:
# your submission, from a fresh clone, as we'll run it (write your repository's URL and remove the #)
# !git clone <your repository URL> /content/submission-check
# !pip install -q -r /content/submission-check/requirements.txt
# !python harness.py evaluate --model /content/submission-check/model.py